In [1]:
import numpy as np
import scipy
from scipy.special import logsumexp
import yaml

# LGSSM Model recap

Recall the LGSSM model: Given $(\rho, \tau)$, we generate a latent trajectory $z_{0:T}$ as 

$$z_0 \sim N(0,1)$$

$$z_t \sim N(\rho z_{t-1}, \tau^2) \hspace{5mm} t=1,...,T$$

The observed trajectory $x_{0:T}$ is given by observing the latent trajectory with pointwise Gaussian noise with variance $\sigma^2$ 

$$x_t \sim N(z_t, \sigma^2) \hspace{5mm} t=0,...,T$$

for some observation noise $\sigma>0$ that we assume is fixed and known. To be explicit, this is a Bayesian model where:

- The Bayesian parameters are $(\rho, \tau, z_{0:T}) \in \mathbb{R}^{T+3}$

- The observed data is $x_{0:T} \in \mathbb{R}^{T+1}$

- The likelihood $\pi(x_{0:T}|\rho, \tau, z_{0:T}) = \pi(x_{0:T}|z_{0:T})$ is independent of the static parameters and is given by $\prod_{t=0}^T \mathcal{N}(x_t|z_t, \sigma^2)$

We place the following hierarchical prior over the Bayesian parameters

$$\pi(\rho, \tau, z_{0:T}) = \pi(\rho, \tau) \pi(z_{0:T}|\rho, \tau)$$

where the prior over static parameters is given by

$$\pi(\rho, \tau) = \pi(\rho)\pi(\tau)$$

where $\pi(\rho) = \text{Unif}(\rho | \rho_{\text{lower}}, \rho_{\text{upper}})$ and $\pi(\tau) = \mathcal{N}_{[\tau_{\text{lower}}, \tau_{\text{upper}}]} (\tau | \tau_{\text{loc}}, \tau_{\text{scale}})$. Given a sample $(\rho, \tau) \sim \pi(\rho, \tau)$, the conditional prior for the latent trajectory is 

$$\pi(z_{0:T}|\rho, \tau) = \mathcal{N}(z_0|0, 1) \prod_{t=1}^T \mathcal{N}(z_t | \rho z_{t-1}, \tau^2)$$

This prior and likelihood define a posterior $\pi(\rho, \tau, z_{0:T}|x_{0:T})$. This posterior can be sampled from using the Kalman filter with Metropolis Hastings, which we shall treat as our ground truth posterior. 

# Conditional volume ratios

The overall goal of this notebook is to compute a notion of volume ratio for the LGSSM posterior relative to the prior. This serves as a measure of posterior concentration, where low volume ratios mean that the posterior is highly concentrated relative to the prior. This is not available in closed form (like it was in the unif_norm experiment), but we can approximate it neatly using the fact that the following two distributions are jointly Gaussian for the LGSSM:

- $\pi(z_{0:T}|\rho, \tau)$ (the prior of $z_{0:T}$ conditional on a fixed $\rho, \tau$).

- $\pi(z_{0:T}|\rho, \tau, x_{0:T})$ (the posterior of $z_{0:T}|x_{0:T}$ conditional on a fixed $\rho, \tau$).

If we write $$\pi(z_{0:T}|\rho, \tau) = \mathcal{N}(z_{0:T}|\mu_{\rho, \tau}^\text{prior}, \Sigma_{\rho, \tau}^\text{prior})$$ and $$\pi(z_{0:T}|\rho, \tau, x_{0:T}) = \mathcal{N}(z_{0:T}|\mu_{\rho, \tau}^\text{posterior}, \Sigma_{\rho, \tau}^\text{posterior})$$

Then, for any $\alpha \in (0,1)$, the $100\cdot \alpha \%$ credible interval of the prior has volume

$$\frac{\pi^{\frac{T+1}{2}}}{\Gamma(\frac{T+1}{2} + 1)} (\chi^2_{T+1}(\alpha))^\frac{T+1}{2} \sqrt{\text{det}(\Sigma_{\rho, \tau}^\text{prior})}$$

Similarly, the $100\cdot \alpha \%$ credible interval of the posterior has volume

$$\frac{\pi^{\frac{T+1}{2}}}{\Gamma(\frac{T+1}{2} + 1)} (\chi^2_{T+1}(\alpha))^\frac{T+1}{2} \sqrt{\text{det}(\Sigma_{\rho, \tau}^\text{posterior})}$$


where $\chi^2_{T+1}(\alpha)$ is the $\alpha$-quantile of the chi-squared distribution with $T+1$ degrees of freedom. Thus, the ratio of the volume of the $100\cdot \alpha \%$ credible intervals is 

$$V_{\rho, \tau} = \frac{\frac{\pi^{\frac{T+1}{2}}}{\Gamma(\frac{T+1}{2} + 1)} (\chi^2_{T+1}(\alpha))^\frac{T+1}{2} \sqrt{\text{det}(\Sigma_{\rho, \tau}^\text{posterior})}}{\frac{\pi^{\frac{T+1}{2}}}{\Gamma(\frac{T+1}{2} + 1)} (\chi^2_{T+1}(\alpha))^\frac{T+1}{2} \sqrt{\text{det}(\Sigma_{\rho, \tau}^\text{prior})}}$$

$$= \sqrt{\frac{\text{det}(\Sigma_{\rho, \tau}^\text{posterior})}{\text{det}(\Sigma_{\rho, \tau}^\text{prior})}}$$

Importantly, this ratio is independent of the choice of $\alpha$.


In [2]:
def prior_covariance(rho, tau, T):
    """
    Conditional on a fixed (rho, tau), this function computes:

    The exact prior covariance matrix of z_{0:T} under AR(1) with z_0 ~ N(0, 1).

    Var(z_0) = 1 (since z_0 ~ N(0, 1))
    Var(z_t) = rho^{2t} + tau^2 * sum_{k=0}^{t-1} rho^{2k}
    Cov(z_i, z_j) = rho^{j-i} * Var(z_i)   for i <= j

    Args:
        rho: AR(1) coefficient
        tau: process noise std
        T: number of time steps
    Returns:
        Sigma_prior: (T+1, T+1) prior covariance matrix of z_{0:T} conditional on a fixed (rho, tau)
    """
    d = T + 1

    # Compute marginal variances
    var = np.zeros(d)
    var[0] = 1.0
    for t in range(1, d):
        var[t] = rho**2 * var[t-1] + tau**2

    # Build covariance matrix using Cov(z_i, z_j) = rho^(j-i) * Var(z_i) for i <= j
    Sigma = np.zeros((d, d))
    for i in range(d):
        for j in range(i, d):
            Sigma[i, j] = rho**(j - i) * var[i]
            Sigma[j, i] = Sigma[i, j]  # symmetry

    return Sigma


def posterior_covariance(rho, tau, sigma_true, T) -> np.ndarray:
    """
    Conditional on a given (rho, tau), this function computes:

    The exact posterior covariance of z_{0:T} | x_{0:T} via Kalman filter + RTS smoother.

    Args:
        rho: AR(1) coefficient
        tau: process noise std
        sigma_true: observation noise std (fixed at sigma_true)
        T: number of steps (z_0, ..., z_T => T+1 variables)
    Returns:
        Sigma_post: (T+1, T+1) posterior covariance matrix
    """
    d = T + 1

    # Kalman Filter forward pass
    P_pred = np.zeros(d) # predicted variances P_{t|t-1}
    P_filt = np.zeros(d) # filtered variances P_{t|t}

    # Initialise with z_0 ~ N(0,1)
    P_pred[0] = 1.0

    for t in range(d):
        # Update step
        S = P_pred[t] + sigma_true**2 # innovation variance
        K = P_pred[t] / S # Kalman gain
        P_filt[t] = (1 - K) * P_pred[t] # filtered variance

        # Predict step
        if t < T:
            P_pred[t+1] = rho**2 * P_filt[t] + tau**2

    # RTS Smoother backward pass
    P_smooth = np.zeros(d)
    G = np.zeros(T) # smoother gains G_t = rho * P_filt[t] / P_pred[t+1]

    P_smooth[T] = P_filt[T]

    for t in range(T-1,-1,-1):
        G[t] = rho * P_filt[t] / P_pred[t+1]
        P_smooth[t] = P_filt[t] + G[t]**2 * (P_smooth[t+1] - P_pred[t+1])

    # Build full (T+1, T+1) covariance matrix:
    
    # Diagonal
    Sigma_post = np.zeros((d, d))
    for t in range(d):
        Sigma_post[t, t] = P_smooth[t]

    # Off-diagonals using the RTS recursion:
    # Cov(z_t, z_{t+1} | x) = G[t] * P_smooth[t+1]
    # Cov(z_t, z_{t+k} | x) = G[t] * Cov(z_{t+1}, z_{t+k} | x)  for k > 1
    for t in range(T):
        Sigma_post[t, t+1] = G[t] * P_smooth[t+1]
        Sigma_post[t+1, t] = Sigma_post[t, t+1]

    for gap in range(2, d):
        for t in range(d - gap):
            Sigma_post[t, t+gap] = G[t] * Sigma_post[t+1, t+gap]
            Sigma_post[t+gap, t] = Sigma_post[t, t+gap]

    return Sigma_post


# Log volume ratio conditional on a (rho, tau) sample
def conditional_log_volume_ratio(rho, tau, sigma_true, T):
    Sigma_prior = prior_covariance(rho, tau, T)
    Sigma_post  = posterior_covariance(rho, tau, sigma_true, T)
    _, logdet_prior = np.linalg.slogdet(Sigma_prior)
    _, logdet_post  = np.linalg.slogdet(Sigma_post)
    log_vr = 0.5 * (logdet_post - logdet_prior)
    return log_vr


# Volume ratio conditional on a (rho, tau) sample
def conditional_volume_ratio(rho, tau, sigma_true, T):
    log_vr = conditional_log_volume_ratio(rho, tau, sigma_true, T)
    vr = np.exp(log_vr)
    return vr

In [3]:
T = 100
rho, tau = 0.9, 0.5
sigma_true = 0.5

log_vr = conditional_log_volume_ratio(rho, tau, sigma_true, T)
vr = np.exp(log_vr)

print(f"Log volume ratio: {log_vr}")
print(f"Volume ratio: {vr}")

Log volume ratio: -46.33286061785988
Volume ratio: 7.549084782220054e-21


# Volume ratios averaged over the prior

The "conditional volume ratios" $V_{\rho, \tau}$ are the volume ratios of the posterior over latent states $z_{0:T}$ conditional on a specific $(\rho, \tau)$. However, since $\rho$ and $\tau$ are random variables, this pointwise volume ratio only defines a notion of concentration for slices across the joint distribution over $(\rho, \tau, z_{0:T})$. To get a better idea of the volume ratio on a global scale, we can average over $(\rho, \tau)$ samples. We have two choices for the sampling distribution over $(\rho, \tau)$: the prior or the posterior. Below, we compute the volume ratio when we average over the prior:

$$V_{\text{prior\_average}} = \int \int V_{\rho, \tau} \pi(\rho, \tau) d\rho d\tau$$

$$\approx \frac{1}{M}\sum_{i=1}^M V_{{\rho_i, \tau_i}}$$

where $(\rho_i, \tau_i) \sim \pi(\rho, \tau)$ for $i=1,...,M$.

In [4]:
def static_params_prior_rv(rho_lower, rho_upper, tau_loc, tau_scale, tau_lower, tau_upper):
    # Prior for static parameters
    rho = scipy.stats.uniform.rvs(loc=rho_lower, scale=rho_upper - rho_lower)
    tau = scipy.stats.truncnorm.rvs(
                a=(tau_lower - tau_loc) / tau_scale,
                b=(tau_upper - tau_loc) / tau_scale,
                loc=tau_loc, scale=tau_scale
                )
    return rho, tau


def log_volume_ratio_prior_average(sigma_true, T, rho_lower, rho_upper,
                     tau_loc, tau_scale, tau_lower, tau_upper, num_samples=1000):
    log_vrs = []
    for samp in range(num_samples):
        # Prior sample
        rho, tau = static_params_prior_rv(rho_lower, rho_upper, tau_loc, tau_scale, tau_lower, tau_upper)
        # Compute log volume ratio for this (rho, tau)
        log_vr = conditional_log_volume_ratio(rho, tau, sigma_true, T)
        # Append to list
        log_vrs.append(log_vr)

    log_mean_vr = logsumexp(log_vrs) - np.log(len(log_vrs))
    return log_mean_vr


def volume_ratio_prior_average(sigma_true, T, rho_lower, rho_upper,
                     tau_loc, tau_scale, tau_lower, tau_upper, num_samples=1000):
    log_mean_vr = log_volume_ratio_prior_average(sigma_true, T, rho_lower, rho_upper,
                     tau_loc, tau_scale, tau_lower, tau_upper, num_samples)
    return np.exp(log_mean_vr)

In [5]:
T = 100
sigma_true = 0.1
rho_lower, rho_upper = 0.0, 1.0
tau_loc, tau_scale, tau_lower, tau_upper = 1.0, 1.0, 0.0, 2.0

log_mean_vr = log_volume_ratio_prior_average(sigma_true, T, rho_lower, rho_upper,
                     tau_loc, tau_scale, tau_lower, tau_upper, num_samples=1000)
mean_vr = np.exp(log_mean_vr)
mean_vr

0.0003878770932403216

# Posterior averaged volume ratios

Below, we compute the volume ratio when we average over the posterior:

$$V_{\text{prior\_average}} = \int \int V_{\rho, \tau} \pi(\rho, \tau|x_{0:T}) d\rho d\tau$$

$$\approx \frac{1}{M}\sum_{i=1}^M V_{{\rho_i, \tau_i}}$$

where $(\rho_i, \tau_i) \sim \pi(\rho, \tau|x_{0:T})$ for $i=1,...,M$.

In [6]:
def log_volume_ratio_posterior_average(sigma_true, T, rho_samples, tau_samples, num_samples=1000):
    log_vrs = []
    for samp in range(num_samples):
        # Posterior sample
        idx = np.random.choice(len(rho_samples))
        rho = rho_samples[idx]
        tau = tau_samples[idx]
        # Compute log volume ratio for this (rho, tau)
        log_vr = conditional_log_volume_ratio(rho, tau, sigma_true, T)
        # Append to list
        log_vrs.append(log_vr)

    log_mean_vr = logsumexp(log_vrs) - np.log(len(log_vrs))
    return log_mean_vr


def volume_ratio_posterior_average(sigma_true, T, rho_samples, tau_samples, num_samples=1000):
    log_mean_vr = log_volume_ratio_posterior_average(sigma_true, T, rho_samples, tau_samples, num_samples)
    return np.exp(log_mean_vr)

In [12]:
results_path = "/Users/Lieve/Documents/Masters Project/SBC-SBI/results/toy_examples/lgssm/"
# Import samples
experiment_ID = 1
posterior_samples_dict_path = results_path + f"kalman_filter/posterior_samples{experiment_ID}.npz"
posterior_samples_dict_config_path = results_path + f"kalman_filter/posterior_samples{experiment_ID}.yaml"

posterior_samples_dict = np.load(posterior_samples_dict_path)
rho_samples = posterior_samples_dict["rho_samples"]
tau_samples = posterior_samples_dict["tau_samples"]
z_samples = posterior_samples_dict["z_samples"]

with open(posterior_samples_dict_config_path, "r") as f:
    posterior_samples_dict_config = yaml.safe_load(f)

x_observed_ID = posterior_samples_dict_config["x_observed_ID"]
total_time = posterior_samples_dict_config["total_time"]
rho_lower = posterior_samples_dict_config["rho_lower"]
rho_upper = posterior_samples_dict_config["rho_upper"]
T = posterior_samples_dict_config["T"]
num_iterations = posterior_samples_dict_config["num_iterations"]
step_sizes = posterior_samples_dict_config["step_sizes"]
rho_0 = posterior_samples_dict_config["rho_0"]
tau_0 = posterior_samples_dict_config["tau_0"]
sigma_true = posterior_samples_dict_config["sigma_true"]
tau_loc = posterior_samples_dict_config["tau_loc"]
tau_scale = posterior_samples_dict_config["tau_scale"]
tau_lower = posterior_samples_dict_config["tau_lower"]
tau_upper = posterior_samples_dict_config["tau_upper"]
rejection_rate = posterior_samples_dict_config["rejection_rate"]

print(f"sigma_true: {sigma_true}")
print(f"T: {T}")

sigma_true: 20
T: 100


In [13]:
log_mean_vr = log_volume_ratio_posterior_average(sigma_true, T, rho_samples, tau_samples, num_samples=1000)
mean_vr = np.exp(log_mean_vr)
mean_vr

0.7144475311385298